In [2]:

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from itertools import product
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from pyod.models.knn import KNN
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns', None)

## DATA PREP

In [19]:
df = pd.read_csv('https://raw.githubusercontent.com/sebastiansossah/TFM/main/data/df.zip', index_col=0)

C:\Users\sebastian sossa\AppData\Local\Temp\ipykernel_15172\2866093463.py:1: DtypeWarning: Columns (3,4,5,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('https://raw.githubusercontent.com/sebastiansossah/TFM/main/data/df.zip', index_col=0)


In [20]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.set_index('Timestamp')
df = df.sort_index()

In [6]:
def create_sliding_windows(df, window_size, stride):
    """
    Crea ventanas deslizantes separando normal vs anomalía
    + metadata para mapeo ventana → evento
    
    Returns:
        windows_train: lista de ventanas normales
        windows_test: lista de ventanas con anomalías
        metadata_train: dict con info de cada ventana train
        metadata_test: dict con info de cada ventana test
    """
    if 'Block_ID' not in df.columns or 'Anomaly_Event' not in df.columns:
        print("Necesitas Block_ID y Anomaly_Event")
        return None, None, None, None
    
    windows_train = []
    windows_test = []
    metadata_train = []
    metadata_test = []
    
    train_idx = 0
    test_idx = 0
    
    # Procesar cada bloque
    for block_id in sorted(df['Block_ID'].unique()):
        block_data = df[df['Block_ID'] == block_id].copy()
        
        # Crear ventanas deslizantes
        for i in range(0, len(block_data) - window_size + 1, stride):
            window = block_data.iloc[i:i+window_size]
            
            # Metadata común
            meta = {
                'block_id': block_id,
                'start_time': window.index[0],
                'end_time': window.index[-1],
                'start_idx_in_block': i,
                'anomaly_event': int(window['Anomaly_Event'].max()),
                'has_anomaly': bool(window['Anomaly'].any()),
                'n_anomaly_records': int(window['Anomaly'].sum())
            }
            
            # Separar train/test
            if window['Anomaly_Event'].max() > 0:
                meta['window_idx'] = test_idx
                meta['split'] = 'test'
                windows_test.append(window)
                metadata_test.append(meta)
                test_idx += 1
            else:
                meta['window_idx'] = train_idx
                meta['split'] = 'train'
                windows_train.append(window)
                metadata_train.append(meta)
                train_idx += 1
    
    print(f"Ventanas train (normales): {len(windows_train)}")
    print(f"Ventanas test (con anomalías): {len(windows_test)}")
    
    if len(windows_train) > 0:
        total_train_records = sum(len(w) for w in windows_train)
        print(f"Registros totales en train: {total_train_records}")
    
    if len(windows_test) > 0:
        total_test_records = sum(len(w) for w in windows_test)
        anomaly_records = sum((w['Anomaly'] == 1).sum() for w in windows_test)
        print(f"Registros totales en test: {total_test_records}")
        print(f"Registros con anomalía en test: {anomaly_records}")
        
        # Info de eventos
        unique_events = set(m['anomaly_event'] for m in metadata_test)
        print(f"Eventos únicos en test: {len(unique_events)}")
    
    return windows_train, windows_test, metadata_train, metadata_test

In [7]:
def prepare_data_model(windows_train, windows_test):
    """
    Prepara datos con normalización diferenciada para sensores y actuadores
    """
    # Definir features por tipo
    continuous_features = [
        'FIT101', 'LIT101',
        'AIT201', 'AIT202', 'AIT203', 'FIT201',
        'AIT301', 'AIT302', 'AIT303', 'DPIT301', 'FIT301', 'LIT301',
        'AIT401', 'AIT402', 'FIT401', 'LIT401',
        'AIT501', 'AIT502', 'AIT503', 'AIT504',
        'FIT501', 'FIT502', 'FIT503', 'FIT504',
        'PIT501', 'PIT502', 'PIT503',
        'FIT601', 'FIT602', 'LIT601', 'LIT602'
    ]
    
    binary_features = [
        'MV101', 'P101', 'P102',
        'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206',
        'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302',
        'P401', 'P402', 'P403', 'P404', 'UV401',
        'MV501', 'MV502', 'MV503', 'MV504', 'P501', 'P502',
        'P601', 'P602', 'P603'
    ]
    
    all_features = continuous_features + binary_features
    
    print(f"Features continuas: {len(continuous_features)}")
    print(f"Features binarias: {len(binary_features)}")
    print(f"Total features: {len(all_features)}")
    
    # Convertir ventanas a arrays
    X_train = np.array([w[all_features].values for w in windows_train])
    X_test = np.array([w[all_features].values for w in windows_test])
    
    print(f"\nX_train shape inicial: {X_train.shape}")
    print(f"X_test shape inicial: {X_test.shape}")
    
    # Separar continuos y binarios
    n_continuous = len(continuous_features)
    
    X_train_cont = X_train[:, :, :n_continuous]
    X_train_bin = X_train[:, :, n_continuous:]
    
    X_test_cont = X_test[:, :, :n_continuous]
    X_test_bin = X_test[:, :, n_continuous:]
    
    # Normalizar solo features continuas
    scaler = StandardScaler()
    
    n_samples_train, window_size, _ = X_train_cont.shape
    X_train_cont_reshaped = X_train_cont.reshape(-1, n_continuous)
    X_train_cont_scaled = scaler.fit_transform(X_train_cont_reshaped)
    X_train_cont = X_train_cont_scaled.reshape(n_samples_train, window_size, n_continuous)
    
    n_samples_test = X_test_cont.shape[0]
    X_test_cont_reshaped = X_test_cont.reshape(-1, n_continuous)
    X_test_cont_scaled = scaler.transform(X_test_cont_reshaped)
    X_test_cont = X_test_cont_scaled.reshape(n_samples_test, window_size, n_continuous)
    
    # Recombinar
    X_train_final = np.concatenate([X_train_cont, X_train_bin], axis=2)
    X_test_final = np.concatenate([X_test_cont, X_test_bin], axis=2)
    
    print(f"\nX_train final shape: {X_train_final.shape}")
    print(f"X_test final shape: {X_test_final.shape}")

    X_train_final = np.array(X_train_final, dtype=np.float32)
    X_test_final = np.array(X_test_final, dtype=np.float32)

    return X_train_final, X_test_final, scaler, all_features

In [21]:
EXPERIMENT_CONFIG = {
    'strides': [5, 15, 30],
    'window_size': 60
}

In [ ]:
# for stride in config['strides']:
#     print(f"\n{'='*80}")
#     print(f"STRIDE = {stride}")
#     print(f"{'='*80}\n")
        
#         # Crear ventanas (solo una vez por stride)
#     windows_train, windows_test, metadata_train, metadata_test = create_sliding_windows(
#             df, 
#             window_size=config['window_size'], 
#             stride=stride
#         )
        
# X_train, X_test, scaler, features = prepare_data_model(windows_train, windows_test)

In [22]:
windows_train, windows_test, metadata_train, metadata_test = create_sliding_windows(
            df, 
            window_size=60, 
            stride=30
        )
        
X_train, X_test, scaler, features = prepare_data_model(windows_train, windows_test)

Ventanas train (normales): 2440
Ventanas test (con anomalías): 1418
Registros totales en train: 146400
Registros totales en test: 85080
Registros con anomalía en test: 43318
Eventos únicos en test: 481
Features continuas: 31
Features binarias: 30
Total features: 61

X_train shape inicial: (2440, 60, 61)
X_test shape inicial: (1418, 60, 61)

X_train final shape: (2440, 60, 61)
X_test final shape: (1418, 60, 61)


In [29]:
X_train = X_train.flatten().reshape(-1, 1)
X_test = X_test.flatten().reshape(-1, 1)

In [ ]:


# param_grid_knn = {
#     "n_neighbors": [5, 10, 20],
#     "method": ["largest", "mean", "median"],  # how distance is aggregated
#     "contamination": [0.005, 0.01, 0.02]
# }


knn = KNN(
    n_neighbors=20,
    method="mean",
    contamination=0.01,
    n_jobs=-1)
knn.fit(X_train)


In [ ]:
    y_pred = knn.predict(X_test)  # 1 = outlier, 0 = inlier

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1_knn:
        best_f1_knn = f1
        best_model_knn = knn
        best_params_knn = {
            "n_neighbors": n_neighbors,
            "method": method,
            "contamination": contamination
        }
        print(f"🚀 New best KNN F1={f1:.4f} | Params={best_params_knn}")

# --- Final evaluation ---
y_pred_knn = best_model_knn.predict(X_test_scaled)
roc_auc_knn = roc_auc_score(y_test, y_pred_knn)
pr_auc_knn = average_precision_score(y_test, y_pred_knn)

print("\n✅ Best KNN parameters:", best_params_knn)
print(classification_report(y_test, y_pred_knn, digits=4))
print(f"ROC-AUC: {roc_auc_knn:.4f}")
print(f"PR-AUC:  {pr_auc_knn:.4f}")

## Isolation Forest

In [5]:

# ISolation forest

param_grid = {
    "contamination": [0.001, 0.005, 0.01, 0.02],
    "max_samples": [128, 256, 512],
    "max_features": [0.5, 0.7, 1.0],
    "n_estimators": [200, 400, 800]
}

best_f1 = 0
best_model = None
best_params = {}

for contamination, max_samples, max_features, n_estimators in product(
        param_grid["contamination"],
        param_grid["max_samples"],
        param_grid["max_features"],
        param_grid["n_estimators"]
):
    iso = IsolationForest(
        n_estimators=n_estimators,
        max_samples=max_samples,
        contamination=contamination,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    iso.fit(X_train_scaled)
    y_pred = (iso.predict(X_test_scaled) == -1).astype(int)

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1:
        best_f1 = f1
        best_model = iso
        best_params = {
            "contamination": contamination,
            "max_samples": max_samples,
            "max_features": max_features,
            "n_estimators": n_estimators
        }
        print(f"New best F1={f1:.4f} | Params={best_params}")


y_pred_final = (best_model.predict(X_test_scaled) == -1).astype(int)

report = classification_report(y_test, y_pred_final, digits=4)
roc_auc = roc_auc_score(y_test, y_pred_final)
pr_auc = average_precision_score(y_test, y_pred_final)

print("\n Best parameters:", best_params)
print("\n Final Isolation Forest (improved benchmark):")
print(report)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")


New best F1=0.0024 | Params={'contamination': 0.001, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0060 | Params={'contamination': 0.001, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 400}
New best F1=0.0231 | Params={'contamination': 0.005, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0531 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0585 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 400}
New best F1=0.0617 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.7, 'n_estimators': 400}
New best F1=0.0702 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.7, 'n_estimators': 800}
New best F1=0.0776 | Params={'contamination': 0.01, 'max_samples': 256, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0807 | Params={'contamination': 0.01, 'max_samples': 512, 'max_features

## OCSVM

In [11]:

param_grid_svm = {
    "kernel": ["rbf", "sigmoid"],
    "nu": [0.001, 0.005, 0.01, 0.05],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1]
}

best_f1_svm = 0
best_model_svm = None
best_params_svm = {}

for kernel, nu, gamma in product(
        param_grid_svm["kernel"],
        param_grid_svm["nu"],
        param_grid_svm["gamma"]
):
    ocsvm = OneClassSVM(kernel=kernel, nu=nu, gamma=gamma)
    ocsvm.fit(X_train_scaled)
    y_pred = (ocsvm.predict(X_test_scaled) == -1).astype(int)

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1_svm:
        best_f1_svm = f1
        best_model_svm = ocsvm
        best_params_svm = {
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma
        }
        print(f" New best OCSVM F1={f1:.4f} | Params={best_params_svm}")

y_pred_svm = (best_model_svm.predict(X_test_scaled) == -1).astype(int)
roc_auc_svm = roc_auc_score(y_test, y_pred_svm)
pr_auc_svm = average_precision_score(y_test, y_pred_svm)
print("\n Best One-Class SVM parameters:", best_params_svm)
print(classification_report(y_test, y_pred_svm, digits=4))
print(f"ROC-AUC: {roc_auc_svm:.4f}")
print(f"PR-AUC:  {pr_auc_svm:.4f}")



 New best OCSVM F1=0.5130 | Params={'kernel': 'rbf', 'nu': 0.001, 'gamma': 'scale'}
 New best OCSVM F1=0.7320 | Params={'kernel': 'rbf', 'nu': 0.001, 'gamma': 0.001}
 New best OCSVM F1=0.7383 | Params={'kernel': 'rbf', 'nu': 0.005, 'gamma': 0.001}
 New best OCSVM F1=0.7486 | Params={'kernel': 'rbf', 'nu': 0.01, 'gamma': 0.001}

 Best One-Class SVM parameters: {'kernel': 'rbf', 'nu': 0.01, 'gamma': 0.001}
              precision    recall  f1-score   support

           0     0.9466    0.9804    0.9632      9952
           1     0.8505    0.6685    0.7486      1659

    accuracy                         0.9358     11611
   macro avg     0.8985    0.8244    0.8559     11611
weighted avg     0.9329    0.9358    0.9326     11611

ROC-AUC: 0.8244
PR-AUC:  0.6159


## KNN

In [14]:


# --- KNN parameter grid ---
param_grid_knn = {
    "n_neighbors": [5, 10, 20],
    "method": ["largest", "mean", "median"],  # how distance is aggregated
    "contamination": [0.005, 0.01, 0.02]
}

best_f1_knn = 0
best_model_knn = None
best_params_knn = {}

for n_neighbors, method, contamination in product(
        param_grid_knn["n_neighbors"],
        param_grid_knn["method"],
        param_grid_knn["contamination"]
):
    knn = KNN(
        n_neighbors=n_neighbors,
        method=method,
        contamination=contamination,
        n_jobs=-1
    )
    knn.fit(X_train_scaled)
    y_pred = knn.predict(X_test_scaled)  # 1 = outlier, 0 = inlier

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1_knn:
        best_f1_knn = f1
        best_model_knn = knn
        best_params_knn = {
            "n_neighbors": n_neighbors,
            "method": method,
            "contamination": contamination
        }
        print(f"🚀 New best KNN F1={f1:.4f} | Params={best_params_knn}")

# --- Final evaluation ---
y_pred_knn = best_model_knn.predict(X_test_scaled)
roc_auc_knn = roc_auc_score(y_test, y_pred_knn)
pr_auc_knn = average_precision_score(y_test, y_pred_knn)

print("\n✅ Best KNN parameters:", best_params_knn)
print(classification_report(y_test, y_pred_knn, digits=4))
print(f"ROC-AUC: {roc_auc_knn:.4f}")
print(f"PR-AUC:  {pr_auc_knn:.4f}")


🚀 New best KNN F1=0.0628 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.005}
🚀 New best KNN F1=0.2950 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.01}
🚀 New best KNN F1=0.6100 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.02}
🚀 New best KNN F1=0.6862 | Params={'n_neighbors': 5, 'method': 'mean', 'contamination': 0.02}

✅ Best KNN parameters: {'n_neighbors': 5, 'method': 'mean', 'contamination': 0.02}
              precision    recall  f1-score   support

           0     0.9313    0.9868    0.9583      9952
           1     0.8771    0.5636    0.6862      1659

    accuracy                         0.9264     11611
   macro avg     0.9042    0.7752    0.8223     11611
weighted avg     0.9236    0.9264    0.9194     11611

ROC-AUC: 0.7752
PR-AUC:  0.5567
